# Explore here

In [ ]:
# Your code here

In [ ]:
cd /workspaces/imagenesclas.Rhonalmendoza/data/raw


In [ ]:
DATA_PATH = "/workspaces/imagenesclas.Rhonalmendoza/data/train"
print("Ruta actual:", DATA_PATH)


In [ ]:
! pip install tensorflow==2.12


Importamos

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
print(tf.__version__)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping


VISUALIZACIÓN DE 9 PERROS Y 9 GATOS

In [ ]:
data_dir = "/workspaces/imagenesclas.Rhonalmendoza/data/train"
dogs_dir = os.path.join(data_dir, "dog")

lista_imagenes_dog = sorted([f for f in os.listdir(dogs_dir) if f.endswith(".jpg")])

imagenes_dog = []
for i in range(9):
    img_path = os.path.join(dogs_dir, lista_imagenes_dog[i])
    img = image.load_img(img_path, target_size=(200,200))
    img_array = image.img_to_array(img) / 255.0
    imagenes_dog.append(img_array)

plt.figure(figsize=(12, 8))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(imagenes_dog[i])
    plt.title("Perro")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
cats_dir = os.path.join(data_dir, "cat")

lista_imagenes_cat = sorted([f for f in os.listdir(cats_dir) if f.endswith(".jpg")])

imagenes_cat = []
for i in range(9):
    img_path = os.path.join(cats_dir, lista_imagenes_cat[i])
    img = image.load_img(img_path, target_size=(200,200))
    img_array = image.img_to_array(img) / 255.0
    imagenes_cat.append(img_array)

plt.figure(figsize=(12, 8))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(imagenes_cat[i])
    plt.title("Gato")
    plt.axis("off")

plt.tight_layout()
plt.show()


GENERADORES DE DATOS

In [ ]:
image_size = (200, 200)

datagentrain = ImageDataGenerator(rescale=1./255)
datagentest = ImageDataGenerator(rescale=1./255)

train_data = datagentrain.flow_from_directory(
    "/workspaces/imagenesclas.Rhonalmendoza/data/train",
    target_size=image_size,
    batch_size=32,
    class_mode='binary'
)

test_data = datagentest.flow_from_directory(
    "/workspaces/imagenesclas.Rhonalmendoza/data/test",
    target_size=image_size,
    batch_size=32,
    class_mode='binary',
    shuffle=False
)


In [ ]:
model = Sequential()

model.add(Conv2D(32, (3,3), activation="relu", padding="same", input_shape=(200,200,3)))
model.add(MaxPool2D(2))

model.add(Conv2D(64, (3,3), activation="relu", padding="same"))
model.add(MaxPool2D(2))

model.add(Conv2D(96, (3,3), activation="relu", padding="same"))
model.add(MaxPool2D(2))

model.add(GlobalAveragePooling2D())

model.add(Dense(64, activation="relu"))
model.add(Dense(1, activation="sigmoid"))

model.compile(optimizer=Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])

model.summary()

In [ ]:
direccion_guardado = "/workspaces/imagenesclas.Rhonalmendoza/models"
os.makedirs(direccion_guardado, exist_ok=True)

print("Carpeta de guardado:", direccion_guardado)

In [ ]:
checkpoint_path = os.path.join(direccion_guardado, "best_dog_cat_model.keras")

checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early = EarlyStopping(
    monitor="val_accuracy",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

ENTRENAMIENTO

In [ ]:
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=10,
    callbacks=[checkpoint, early]
)


MODELO VGG16

In [ ]:
model = Sequential()

model.add(Conv2D(32, (3,3), padding="same", activation="relu", input_shape=(128,128,3)))
model.add(Conv2D(32, (3,3), padding="same", activation="relu"))
model.add(MaxPool2D((2,2)))

model.add(Conv2D(64, (3,3), padding="same", activation="relu"))
model.add(Conv2D(64, (3,3), padding="same", activation="relu"))
model.add(MaxPool2D((2,2)))

model.add(Conv2D(128, (3,3), padding="same", activation="relu"))
model.add(Conv2D(128, (3,3), padding="same", activation="relu"))
model.add(MaxPool2D((2,2)))

model.add(Flatten())
model.add(Dense(512, activation="relu"))
model.add(Dense(2, activation="softmax"))

model.compile(optimizer=Adam(1e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
final_model_path = os.path.join(direccion_guardado, "final_dog_cat_model.keras")
model.save(final_model_path)

print("Modelo guardado exitosamente en:", final_model_path)
